In [1]:
import sys, os, glob as _g, subprocess
try:
    import onnxruntime; print(f'ort {onnxruntime.__version__} ready')
except ImportError:
    wdirs = {os.path.dirname(w) for w in _g.glob('/kaggle/input/**/*.whl', recursive=True)
             if 'onnxruntime' in os.path.basename(w)}
    if not wdirs: raise RuntimeError('no onnxruntime wheel in /kaggle/input')
    fl = [x for d in wdirs for x in ['--find-links', d]]
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--no-index', *fl, 'onnxruntime'], check=True)
    import onnxruntime; print(f'ort {onnxruntime.__version__} installed')


ort 1.24.4 installed


# BirdCLEF 2026 Inference v35 — PerchTransformer

**Strategy**: Run the 5-fold PerchTransformer (TransformerEncoder head) trained
from scratch in v35. Average sigmoid predictions across folds.

**Score history**
| Version | LB |
|---------|----|
| v27 GRU-only | 0.872 |
| v29 GRU fine-tune | 0.874 |
| v30 GRU fine-tune | **0.875** |
| v33 GRU+SSM | 0.875 |
| v35 Transformer | ? |

**Kaggle inputs required**
1. `birdclef-2026`
2. `chiragggg/birdclef-2026-perch-onnx`  (or `rishikeshjani/perch-onnx-for-birdclef-2026`)
3. `chiragggg/birdclef-2026-perch-weights-v35-transformer`


In [2]:
import os, warnings, gc
from pathlib import Path
import numpy as np, pandas as pd, soundfile as sf, librosa, onnxruntime as ort
from scipy.ndimage import gaussian_filter1d
import torch, torch.nn as nn
from torch.cuda.amp import autocast
from tqdm import tqdm
warnings.filterwarnings('ignore')

CFG = dict(
    folds=5,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    # Perch ONNX params
    perch_sr=32000, perch_seconds=5, perch_emb_dim=1536, perch_batch=16,
    # Transformer architecture — must match v35 training exactly
    tf_d_model=512,
    tf_nhead=8,
    tf_layers=2,
    tf_ffn_dim=1024,
    tf_dropout=0.1,
    tf_max_seq=24,
    gauss_sigma=1.0,
)
CFG['perch_target'] = CFG['perch_sr'] * CFG['perch_seconds']  # 160000
device = torch.device(CFG['device'])
torch.set_num_threads(os.cpu_count() or 4)
print(f'Device: {device}  ort: {ort.__version__}')
print(f'PerchTransformer v35  d_model={CFG["tf_d_model"]}  nhead={CFG["tf_nhead"]}  layers={CFG["tf_layers"]}')
print(f'gauss_sigma={CFG["gauss_sigma"]}')


Device: cpu  ort: 1.24.4
GRU+SSM v33  hidden=512  layers=2  ssm_kernel=5
gru_weight=1.0  gauss_sigma=1.0  (mel branch disabled)


In [3]:
def _fe(*c):
    return next((p for p in c if os.path.exists(p)), c[0])

TAXONOMY_CSV = _fe('/kaggle/input/birdclef-2026/taxonomy.csv',
                   '/kaggle/input/competitions/birdclef-2026/taxonomy.csv')
TEST_AUDIO   = _fe('/kaggle/input/birdclef-2026/test_soundscapes',
                   '/kaggle/input/competitions/birdclef-2026/test_soundscapes')
SAMPLE_SUB   = _fe('/kaggle/input/birdclef-2026/sample_submission.csv',
                   '/kaggle/input/competitions/birdclef-2026/sample_submission.csv')

# v35 Transformer checkpoints
TXF_CKPT_DIR = _fe('/kaggle/input/birdclef-2026-perch-weights-v35-transformer',
                   '/kaggle/input/datasets/chiragggg/birdclef-2026-perch-weights-v35-transformer',
                   '/kaggle/working')

ONNX_PATH = None
for _c in [
    '/kaggle/input/birdclef-2026-perch-onnx/perch_v2_cpu.onnx',
    '/kaggle/input/datasets/chiragggg/birdclef-2026-perch-onnx/perch_v2_cpu.onnx',
    '/kaggle/input/perch-onnx-for-birdclef2026/perch_v2_cpu.onnx',
    '/kaggle/input/datasets/rishikeshjani/perch-onnx-for-birdclef-2026/perch_v2.onnx',
]:
    if os.path.exists(_c):
        ONNX_PATH = _c
        break

taxonomy_df = pd.read_csv(TAXONOMY_CSV)
species     = taxonomy_df['primary_label'].astype(str).tolist()
n_classes   = len(species)
sp_idx      = {l: i for i, l in enumerate(species)}
print(f'Species       : {n_classes}')
print(f'TXF_CKPT_DIR  : {TXF_CKPT_DIR}')
print(f'ONNX_PATH     : {ONNX_PATH}')


Species      : 234
GRU_CKPT_DIR : /kaggle/input/datasets/chiragggg/birdclef-2026-perch-weights-v33-hybrid
ONNX_PATH    : /kaggle/input/datasets/rishikeshjani/perch-onnx-for-birdclef-2026/perch_v2.onnx


In [4]:
class PerchTransformer(nn.Module):
    """v35 architecture — must match training exactly."""
    def __init__(self, n_classes, emb_dim=1536, d_model=512, nhead=8,
                 num_layers=2, ffn_dim=1024, dropout=0.1):
        super().__init__()
        self.proj = nn.Sequential(
            nn.LayerNorm(emb_dim),
            nn.Linear(emb_dim, d_model),
            nn.GELU(),
        )
        self.pos_emb = nn.Embedding(CFG['tf_max_seq'] + 1, d_model)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=ffn_dim, dropout=dropout,
            batch_first=True, norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Dropout(0.2),
            nn.Linear(d_model, n_classes),
        )

    def forward(self, x, src_key_padding_mask=None):
        single = (x.dim() == 2)
        if single:
            x = x.unsqueeze(1)
        B, T, _ = x.shape
        z   = self.proj(x)
        pos = torch.arange(T, device=x.device).unsqueeze(0)
        z   = z + self.pos_emb(pos)
        h   = self.transformer(z, src_key_padding_mask=src_key_padding_mask)
        out = self.head(h)
        return out.squeeze(1) if single else out


print('PerchTransformer defined')


PerchGRUSSM defined


In [5]:
def _load_txf(names, ckpt_dir):
    ms = []
    for n in names:
        p = Path(ckpt_dir) / n
        if not p.exists():
            print(f'  MISSING: {p}')
            continue
        m = PerchTransformer(
            n_classes, emb_dim=CFG['perch_emb_dim'],
            d_model=CFG['tf_d_model'], nhead=CFG['tf_nhead'],
            num_layers=CFG['tf_layers'], ffn_dim=CFG['tf_ffn_dim'],
            dropout=CFG['tf_dropout'],
        ).to(device)
        m.load_state_dict(torch.load(p, map_location=device, weights_only=True), strict=True)
        m.eval()
        ms.append(m)
        print(f'  OK {n}')
    return ms

print('Loading v35 Transformer checkpoints...')
txf_models = _load_txf(
    [f'perch_tf_v35_fold{i}.pt' for i in range(CFG['folds'])],
    TXF_CKPT_DIR,
)
print(f'Loaded Transformer: {len(txf_models)}/5')
if len(txf_models) == 0:
    print('WARNING: no v35 checkpoints found -- check TXF_CKPT_DIR and checkpoint names.')


Loading v33 GRU+SSM checkpoints...
  OK perch_gru_ssm_v33_fold0.pt
  OK perch_gru_ssm_v33_fold1.pt
  OK perch_gru_ssm_v33_fold2.pt
  OK perch_gru_ssm_v33_fold3.pt
  OK perch_gru_ssm_v33_fold4.pt
Loaded GRU+SSM: 5/5


In [6]:
_sess = None; _inp = None; _eidx = 0; _onnx_ok = False
if ONNX_PATH is None:
    print('ONNX not found -- GRU branch will be disabled')
else:
    try:
        opts = ort.SessionOptions()
        opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
        opts.intra_op_num_threads = os.cpu_count() or 4
        _sess = ort.InferenceSession(ONNX_PATH, sess_options=opts,
                                     providers=['CPUExecutionProvider'])
        _inp  = _sess.get_inputs()[0].name
        _out_names = [o.name for o in _sess.get_outputs()]
        ekey  = next((o.name for o in _sess.get_outputs()
                      if o.shape and o.shape[-1] == 1536), _out_names[0])
        _eidx = _out_names.index(ekey)
        _t    = _sess.run(None, {_inp: np.zeros((1, CFG['perch_target']), np.float32)})
        _e    = _t[_eidx]
        if _e.ndim == 3:
            _e = _e.mean(1)
        assert _e.shape[-1] == 1536, f'Expected 1536-d emb, got {_e.shape}'
        _onnx_ok = True
        print(f'ONNX OK: {Path(ONNX_PATH).name}  ekey={ekey}  emb={_e.shape}')
    except Exception as ex:
        print(f'ONNX ERROR: {ex}')


ONNX OK: perch_v2.onnx  ekey=embedding  emb=(1, 1536)


In [7]:
_amp = (device.type == 'cuda')


def _embs(path, ends):
    """Run Perch ONNX on audio -> {end_sec: emb_1536} dict."""
    if not _onnx_ok or not ends:
        return {}
    try:
        y, sr = sf.read(path, always_2d=False)
        if y.ndim == 2:
            y = y.mean(1)
        if sr != CFG['perch_sr']:
            y = librosa.resample(y.astype(np.float32), orig_sr=sr, target_sr=CFG['perch_sr'])
        y = y.astype(np.float32)
    except Exception as e:
        print(f'[W] perch read: {e}')
        return {}
    clips = []
    for es in ends:
        e0 = int(es * CFG['perch_sr'])
        s0 = max(0, e0 - CFG['perch_target'])
        c  = y[s0:e0]
        if len(c) < CFG['perch_target']:
            c = np.pad(c, (0, CFG['perch_target'] - len(c)))
        clips.append(c)
    all_embs = []
    for bi in range(0, len(clips), CFG['perch_batch']):
        B   = np.stack(clips[bi:bi + CFG['perch_batch']])
        out = _sess.run(None, {_inp: B})[_eidx]
        if out.ndim == 3:
            out = out.mean(1)
        all_embs.append(out.astype(np.float32))
    return dict(zip(ends, np.vstack(all_embs)))


def predict_v35(path, ends):
    """PerchTransformer single-pass prediction. Returns (T, n_classes) probability array."""
    T = len(ends)
    if not txf_models or not _onnx_ok:
        return np.full((T, n_classes), 0.5, np.float32)

    em  = _embs(path, ends)
    seq = torch.from_numpy(
        np.stack([em.get(e, np.zeros(CFG['perch_emb_dim'], np.float32)) for e in ends])
    ).float().unsqueeze(0).to(device)  # (1, T, 1536)

    preds = []
    for m in txf_models:
        with torch.inference_mode(), autocast(enabled=_amp):
            preds.append(torch.sigmoid(m(seq).float())[0].cpu().numpy())

    p = np.mean(preds, axis=0)  # (T, n_classes) — fold average

    if T > 1 and CFG['gauss_sigma'] > 0:
        p = gaussian_filter1d(p.astype(np.float64), sigma=CFG['gauss_sigma'], axis=0).astype(np.float32)
    return p


print(f'predict_v35 defined  Transformer folds={len(txf_models)}  onnx_ok={_onnx_ok}')


predict_v33 defined  GRU+SSM folds=5  onnx_ok=True


In [8]:
sub = pd.read_csv(SAMPLE_SUB).copy()
sub['_sc'] = sub['row_id'].str.rsplit('_', n=1).str[0]
print(f'Rows: {len(sub)}')

row_ids = []; probs_list = []; n_miss = 0; n_err = 0

for sc, grp in tqdm(sub.groupby('_sc'), desc='soundscapes', unit='f'):
    rids = [str(r) for r in grp['row_id']]
    ap = None
    for ext in ['.ogg', '.wav', '.flac']:
        c = Path(TEST_AUDIO) / f'{sc}{ext}'
        if c.exists():
            ap = str(c)
            break
    if ap is None:
        n_miss += 1
        row_ids.extend(rids)
        probs_list.append(np.full((len(rids), n_classes), 0.5, np.float32))
        continue
    try:
        ends = [int(r.rsplit('_', 1)[-1]) for r in rids]
    except Exception:
        n_err += 1
        row_ids.extend(rids)
        probs_list.append(np.full((len(rids), n_classes), 0.5, np.float32))
        continue
    try:
        p = predict_v35(ap, ends)
        row_ids.extend(rids)
        probs_list.append(p)
    except Exception as e:
        n_err += 1
        print(f'ERR {sc}: {e}')
        row_ids.extend(rids)
        probs_list.append(np.full((len(rids), n_classes), 0.5, np.float32))

print(f'Done  missing={n_miss}  errors={n_err}')


Rows: 3


soundscapes: 100%|██████████| 1/1 [00:00<00:00, 70.15f/s]

Done  missing=1  errors=0


In [9]:
mat = np.concatenate(probs_list, axis=0)
mu, sd = mat.mean(), mat.std()
if abs(mu - 0.5) < 0.001 and sd < 0.01:
    print(f'WARNING: all-neutral predictions (mean={mu:.4f}, std={sd:.4f})')
    print('  Check Cell 6 (ONNX) and Cell 5 (checkpoints).')
else:
    print(f'OK  mean={mu:.4f}  std={sd:.4f}')

sub_df = pd.DataFrame(mat, columns=species)
sub_df.insert(0, 'row_id', row_ids)
cols   = pd.read_csv(SAMPLE_SUB, nrows=0).columns.tolist()
sub_df = sub_df[cols]
sub_df.to_csv('/kaggle/working/submission.csv', index=False)
print(f'Saved  shape={sub_df.shape}')
print(f'v35 PerchTransformer  folds={len(txf_models)}  gauss_sigma={CFG["gauss_sigma"]}')
sub_df.head(3)


  Check Cell 6 (ONNX) and Cell 5 (checkpoints).
Saved  shape=(3, 235)
v33 GRU+SSM  gru_weight=1.0  folds=5  gauss_sigma=1.0


,row_id,1161364,116570,1176823,1491113,1595929,209233,22930,22956,22961,...,whnjay1,whtdov,whwpic1,y00678,yebcar,yebela1,yecmac,yecpar,yehcar1,yeofly1
0,BC2026_Test_0001_S05_20250227_010002_5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
1,BC2026_Test_0001_S05_20250227_010002_10,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
2,BC2026_Test_0001_S05_20250227_010002_15,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
